# Architectural variations in Physics-Informed Neural Networks
## SIREN & Fourier-feature networks in JAX, with a Clustering Analysis

**Author:** Aleksa &nbsp;|&nbsp; **Framework:** [JAX](https://jax.readthedocs.io)

---

### What this project is about

Physics-Informed Neural Networks (**PINNs**) solve differential equations by training a
neural network to satisfy the equation directly, using automatic differentiation for the
PDE residual. They are conceptually elegant but notoriously hard to train: their accuracy
depends *heavily* on the **architecture of the network** itself.

This project follows the perspective of Ben Moseley's
[`scalable-pinns-workshop`](https://github.com/benmoseley/scalable-pinns-workshop) and the
[`FBPINNs`](https://github.com/benmoseley/FBPINNs) library — specifically the `networks.py`
module, where the network is a swappable component (`FCN`, `SIREN`, `FourierFCN`, ...).
Rather than re-implementing a full PINN framework (no need, in the era of mature libraries),
we **isolate the network architecture** and ask:

> *How do the input embedding and the activation change what a PINN can learn,
> and where do these tricks break down?*

### Architectures implemented from scratch

We study three architectures, written from scratch in small, functional JAX (mirrors
`networks.py`):

1. **Plain MLP** with `tanh` activations — the textbook PINN baseline.
2. **SIREN** — sinusoidal representation networks (Sitzmann et al., 2020); hyperparameter
   $\omega_0$, special initialisation.
3. **Fourier-feature network** — random Fourier embedding (Tancik et al., 2020) feeding an
   MLP; bandwidth $\sigma$.

### Course-baseline component

The end of the notebook ties the architectural study with a
**clustering analysis of solution space**: across all the (architecture, hyperparameter)
runs we perform, every trained PINN produces a function $u(t)$ that we sample on a fixed
grid. These solution vectors live in a high-dimensional space; we

- reduce them to two dimensions with **PCA implemented from scratch**,
- partition them with **k-means implemented from scratch**,

and visualise the *regimes* the PINN falls into (well-fit, under-fit due to spectral bias,
unstable / ringing). 

### Problems

- **High-frequency 1D damped harmonic oscillator** — exposes *spectral bias*.
- **1D viscous Burgers' equation** — develops a shock, probes architectural *limits*.

### Roadmap

| Section | Content |
|---|---|
| 1 | Background: PINNs, spectral bias, SIREN, Fourier features |
| 2 | The network zoo (`networks.py`-style, JAX, from scratch) |
| 3 | Experiment A — damped oscillator + spectral-bias sweep |
| 4 | Experiment B — Burgers' equation (shock) |
| 5 | Hyperparameter ablations ($\omega_0$, $\sigma$) |
| **6** | **PRML coursework baseline: PCA + k-means clustering of the solution corpus** |
| 7 | Limits, link to FBPINNs, conclusions |

> **Running.** Everything depends only on `jax`, `numpy`, `matplotlib`. Runs on CPU; a GPU
> (Colab T4, faculty cluster) makes Section 4 much faster. The training budgets in this
> version are deliberately lighter than the long-form variant so the whole notebook fits
> in a reasonable CPU run.


## 1. Background

### 1.1 What is a PINN?

For a differential equation $\mathcal{N}[u](\mathbf{x}) = 0$ on $\Omega$ with boundary /
initial constraints $\mathcal{B}[u] = 0$, a PINN represents the solution by a neural
network $u_\theta(\mathbf{x})$ and minimises

$$
\mathcal{L}(\theta) =
\tfrac{1}{N_r}\sum_i \big\|\mathcal{N}[u_\theta](\mathbf{x}_i^r)\big\|^2
\;+\;
\lambda\,\tfrac{1}{N_b}\sum_j \big\|\mathcal{B}[u_\theta](\mathbf{x}_j^b)\big\|^2 .
$$

No labelled solution data is required — the differential operator is evaluated *exactly*
with automatic differentiation. Collocation points $\mathbf{x}_i^r$ are sampled inside
$\Omega$; boundary/initial points enforce the constraints.

### 1.2 Spectral bias

Standard fully-connected `tanh`-MLPs exhibit **spectral bias** (Rahaman et al., 2019):
under gradient descent they learn low frequencies first and high frequencies very slowly.
For PINNs this is fatal — physically interesting solutions are often high-frequency or
multi-scale, and the PDE residual involves *derivatives* that amplify high-frequency
error.

### 1.3 SIREN

SIRENs (Sitzmann et al., 2020) replace `tanh` with $\sin$:
$\phi_\ell(\mathbf{z}) = \sin\!\big(\omega_0\,(W_\ell\mathbf{z}+b_\ell)\big).$
Two ingredients: a frequency scale $\omega_0$ at the first layer, and a specific
initialisation (first layer $\mathcal{U}(-1/n,1/n)$, then
$\mathcal{U}(-\sqrt{6/n}/\omega_0,\,\sqrt{6/n}/\omega_0)$) that keeps activations stable
through depth. Derivatives of a SIREN are SIRENs — pleasant for high-order PDE residuals.

### 1.4 Fourier features

Instead of changing the activation, change the *input embedding* (Tancik et al., 2020).
With a fixed random matrix $B\in\mathbb{R}^{d\times m}$, $B_{ij}\sim\mathcal{N}(0,\sigma^2)$:
$\gamma(\mathbf{x}) = \big[\sin(2\pi B^\top\mathbf{x}),\ \cos(2\pi B^\top\mathbf{x})\big]$,
then feed $\gamma(\mathbf{x})$ to an ordinary `tanh`-MLP. The bandwidth $\sigma$ controls
the injected frequency range and is *problem-dependent* — too small and spectral bias
remains, too large and derivatives become noisy.


## 2. Setup and the network zoo

Following the *functional* design of `fbpinns/networks.py`: a network = a pair of pure
functions (initialise params, apply on a single input vector). We `vmap` over batches
and let `jax.grad` build derivatives.

In [1]:
import jax
import jax.numpy as jnp
from jax import random, grad, vmap, jit
import numpy as np
import matplotlib.pyplot as plt

jax.config.update("jax_enable_x64", True)

print("JAX version :", jax.__version__)
print("Devices     :", jax.devices())
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

# Global solution corpus: every trained PINN appends its u(t) on a fixed grid here,
# together with metadata. Section 6 consumes this.
T_CORPUS = np.linspace(0.0, 1.0, 200)
CORPUS = {"X": [], "meta": []}    # X: list of np.ndarray(200,); meta: list of dict

JAX version : 0.10.0
Devices     : [CpuDevice(id=0)]


### 2.1 Initialisers and forward passes

Each `make_*` returns `(params, apply)` with `apply(params, x)` mapping `x:(d,)` to
`(out,)`. The Fourier matrix `B` is captured in a closure and **fixed** (random-feature
formulation).

In [2]:
#  plain fully-connected network (tanh)
def glorot_init(key, sizes):
    # Glorot/Xavier init for a tanh-MLP. sizes = [in, h1, ..., out]
    params = []
    keys = random.split(key, len(sizes) - 1)
    for k, (nin, nout) in zip(keys, zip(sizes[:-1], sizes[1:])):
        std = jnp.sqrt(2.0 / (nin + nout))
        W = std * random.normal(k, (nin, nout))
        b = jnp.zeros((nout,))
        params.append((W, b))
    return params

def mlp_apply(params, x):
    a = x
    for W, b in params[:-1]:
        a = jnp.tanh(a @ W + b)
    W, b = params[-1]
    return a @ W + b

def make_mlp(key, sizes):
    return glorot_init(key, sizes), mlp_apply


# SIREN 
def siren_init(key, sizes, w0_first=30.0, w0=1.0):
    params = []
    keys = random.split(key, len(sizes) - 1)
    for i, (k, (nin, nout)) in enumerate(zip(keys, zip(sizes[:-1], sizes[1:]))):
        lim = 1.0 / nin if i == 0 else np.sqrt(6.0 / nin) / w0
        W = random.uniform(k, (nin, nout), minval=-lim, maxval=lim)
        b = jnp.zeros((nout,))
        params.append((W, b))
    return params

def siren_apply(params, x, w0_first=30.0, w0=1.0):
    W, b = params[0]
    a = jnp.sin(w0_first * (x @ W + b))
    for W, b in params[1:-1]:
        a = jnp.sin(w0 * (a @ W + b))
    W, b = params[-1]
    return a @ W + b

def make_siren(key, sizes, w0_first=30.0, w0=1.0):
    params = siren_init(key, sizes, w0_first, w0)
    return params, lambda p, x: siren_apply(p, x, w0_first, w0)


# Fourier-feature network 
def make_fourier(key, in_dim, n_features, sigma, hidden):
    # Random Fourier features + tanh-MLP. hidden = [h1, ..., out]
    bkey, fkey = random.split(key)
    B = sigma * random.normal(bkey, (in_dim, n_features))   # FIXED
    sizes = [2 * n_features] + list(hidden)
    params = glorot_init(fkey, sizes)
    def apply(p, x):
        proj = 2.0 * jnp.pi * (x @ B)
        feats = jnp.concatenate([jnp.sin(proj), jnp.cos(proj)], axis=-1)
        return mlp_apply(p, feats)
    return params, apply

### 2.2 A minimal Adam optimiser (PyTree-based)

In [3]:
def adam_init(params):
    m = jax.tree_util.tree_map(jnp.zeros_like, params)
    v = jax.tree_util.tree_map(jnp.zeros_like, params)
    return (m, v, jnp.array(0, dtype=jnp.int64))

def adam_step(params, grads, state, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8):
    m, v, t = state
    t = t + 1
    m = jax.tree_util.tree_map(lambda m, g: b1 * m + (1 - b1) * g, m, grads)
    v = jax.tree_util.tree_map(lambda v, g: b2 * v + (1 - b2) * g * g, v, grads)
    bc1 = 1 - b1 ** t
    bc2 = 1 - b2 ** t
    params = jax.tree_util.tree_map(
        lambda p, m, v: p - lr * (m / bc1) / (jnp.sqrt(v / bc2) + eps),
        params, m, v)
    return params, (m, v, t)